# S6 - Feature engineering

Raw BTS columns → model features. Horizon: **we predict before departure**, so no "at-departure / post-flight" column may enter as a feature (leakage).

This notebook is a **prototype** - once the features stabilize, the logic moves to `src/flight_delay/transform/features.py` as functions (Phase B just wraps them).

In [8]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA = PROJECT_ROOT / "data" / "processed"

df = pd.read_parquet(DATA)
flights = df.loc[(df["Cancelled"] == False) & (df["Diverted"] == False), :].copy().reset_index(drop=True)
flights["y"] = flights["ArrDelay"].ge(15).astype(int)
flights.shape

(6965247, 33)

## Step 1 - column whitelist (leakage boundary)

With leakage it is safer to list what is **allowed** than what is forbidden: a new column drops out by default instead of sneaking in.
Keep `FlightDate` aside - for the time split (S8) and historical features, not as a raw feature. Skip `Tail_Number` for now (high cardinality; may be useful for aircraft-level history later).

In [9]:
# Columns known BEFORE departure - the only ones allowed as features.
SAFE_RAW = [
    "Month", "DayOfWeek",
    "Reporting_Airline",
    "Origin", "Dest", "OriginState", "DestState",
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime",
    "Distance", "DistanceGroup",
]

# At-departure / post-flight columns - LEAKAGE, never features.
LEAKAGE = [
    "DepDelay", "DepDelayMinutes", "DepDel15",
    "TaxiOut", "TaxiIn",
    "ActualElapsedTime", "AirTime",
    "ArrDelay", "ArrDelayMinutes", "ArrDel15",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
    "Cancelled", "Diverted", "CancellationCode",
]

# Sanity-check 1: nothing is both safe and leakage.
assert not (set(SAFE_RAW) & set(LEAKAGE))

# Sanity-check 2: what is left unassigned? (expected: FlightDate, Tail_Number, y)
set(flights.columns) - set(SAFE_RAW) - set(LEAKAGE)

{'FlightDate', 'Tail_Number', 'y'}

## Step 2 - "free" features (straight from raw columns, no aggregation)

Only `DepHour` is **genuinely needed**: `CRSDepTime` is HHMM as a number (1345, 900, 2400) - as an int it is not linear in time (1345→1400 is a jump of 55, not 15), so we decode it to the hour and fix `2400→0`.

`IsWeekend` / `Season` are **partly redundant for XGBoost** (the tree splits `DayOfWeek` / `Month` itself) - but they help the linear baseline in S7, which cannot find thresholds on its own. We build them so both models have what they need.

> `Distance` and `DistanceGroup` are redundant (one is the binned other) - we pick one in S7/S8.

In [10]:
feat = flights[SAFE_RAW].copy()

# Scheduled hour: CRSDepTime = HHMM; 2400 (midnight) -> 0.
feat["DepHour"] = (flights["CRSDepTime"] // 100).replace(24, 0)

# Weekend (DayOfWeek: 1 = Monday ... 7 = Sunday).
feat["IsWeekend"] = flights["DayOfWeek"].isin([6, 7]).astype(int)

# Season from month.
feat["Season"] = flights["Month"].map({
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "fall", 10: "fall", 11: "fall",
})

feat["y"] = flights["y"]
feat.head()

,Month,DayOfWeek,Reporting_Airline,Origin,Dest,OriginState,DestState,CRSDepTime,CRSArrTime,CRSElapsedTime,Distance,DistanceGroup,DepHour,IsWeekend,Season,y
0,1,1,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
1,1,2,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
2,1,3,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
3,1,4,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,1
4,1,5,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0


## Step 3 - encoding: native categorical

Decision: for high cardinality (`Origin` / `Dest` / `Reporting_Airline`) use **native categorical** in XGBoost (`enable_categorical=True`): no leakage, no column blow-up, optimal subset splits. We don't hard-encode anything - we only set the right dtype and let the model handle it in S8. (Comparison of one-hot / ordinal / native / target encoding: see the chat discussion.)

In [11]:
CATEGORICAL = ["Origin", "Dest", "OriginState", "DestState", "Reporting_Airline", "Season"]
feat[CATEGORICAL] = feat[CATEGORICAL].astype("category")

feat.dtypes

Month                    int8
DayOfWeek                int8
Reporting_Airline    category
Origin               category
Dest                 category
OriginState          category
DestState            category
CRSDepTime              int16
CRSArrTime              int16
CRSElapsedTime        float32
Distance              float32
DistanceGroup            int8
DepHour                 int16
IsWeekend               int64
Season               category
y                       int64
dtype: object

## Step 4 - historical delay-rate features (leakage-safe target encoding)

The categorical identity tells XGBoost *which* airport / carrier a flight is; a **historical delay rate** tells it *how bad that airport / carrier has been so far*. This is target encoding - powerful, but the classic leakage trap: averaging `y` over the whole set (including the row itself and the future) lets the model peek at the answer.

**Leakage-safe by construction (expanding by `FlightDate`):** for a flight on day $$t$$ we use only earlier days $$s < t$$:
$$h_{c,t} = \dfrac{\sum_{s<t}\ \sum_{i \in c,\ \text{date}=s} y_i}{\sum_{s<t}\ \sum_{i \in c} 1}$$

**Smoothing (shrinkage)** - a rare category gives a noisy rate ($$\operatorname{Var}(\hat p) = \hat p (1-\hat p)/n$$), so we pull it toward the global prior:
$$\bar y_c = \dfrac{n_c\,\hat p_c + m\,\bar y_{\text{global}}}{n_c + m}$$
Nice side effect: a brand-new category ($$n_c = 0$$) collapses to the prior, so **cold start needs no NaN patching**.

Computing this over the **whole** frame is still safe under the S8 time split: a future (test) row only ever pulls history from earlier days - which may include train days, exactly what production sees - while a train row never sees the future. The only crumb of leakage is the global prior being one constant over the full set; negligible and standard to accept.

Implementation trick: never loop per row (that would be $$O(n^2)$$). Collapse to one row per `(key, date)`, take a cumulative sum over dates, subtract the current day, then map back.

In [ ]:
import numpy as np

PRIOR = float(flights["y"].mean())
M = 100  


def add_history_feature(frame, keys, m=M, prior=PRIOR):
    """Leakage-safe historical delay rate for `keys`, using only days strictly before each flight.

    Smoothed toward `prior` with strength `m`. Returns a float32 array aligned to `frame`'s rows.
    `keys`: a column name or a list (e.g. "Origin", or ["Origin", "Dest"] for a route).
    """
    if isinstance(keys, str):
        keys = [keys]
    daily = (
        frame.groupby(keys + ["FlightDate"], observed=True)["y"]
        .agg(sum_y="sum", n="count")
        .reset_index()
        .sort_values("FlightDate")
    )
    g = daily.groupby(keys, observed=True)
    daily["hist_sum"] = g["sum_y"].cumsum() - daily["sum_y"]
    daily["hist_n"] = g["n"].cumsum() - daily["n"]
    daily["hist_rate"] = (daily["hist_sum"] + m * prior) / (daily["hist_n"] + m)
    out = frame[keys + ["FlightDate"]].merge(
        daily[keys + ["FlightDate", "hist_rate"]], on=keys + ["FlightDate"], how="left"
    )
    return out["hist_rate"].to_numpy(dtype="float32")


feat["CarrierHist"] = add_history_feature(flights, "Reporting_Airline")
feat["OriginHist"] = add_history_feature(flights, "Origin")
feat["RouteHist"] = add_history_feature(flights, ["Origin", "Dest"])

feat[["CarrierHist", "OriginHist", "RouteHist"]].describe()

,CarrierHist,OriginHist,RouteHist
count,6.965247e+06,6.965247e+06,6.965247e+06
mean,2.158489e-01,2.160503e-01,2.149356e-01
std,4.299799e-02,4.143675e-02,5.361684e-02
min,4.898693e-02,5.718584e-02,5.831360e-02
25%,1.888810e-01,1.871302e-01,1.773074e-01
50%,2.115710e-01,2.094301e-01,2.108241e-01
75%,2.419452e-01,2.411077e-01,2.470757e-01
max,3.568850e-01,4.102055e-01,4.922233e-01


## Step 4c - airport-at-hour signals (congestion + finer history)

Two features keyed on `(Origin, DepHour)`:
- **`OriginHourCongestion`** - how many flights are *scheduled* to depart from this airport in this hour on this day. Built only from the schedule (`CRSDepTime`), so **no leakage**. This is the hub-congestion signal EDA2 found (busy hub hours delay more), handed to the model directly instead of hoping it rediscovers it from raw columns.
- **`OriginHourHist`** - the leakage-safe historical delay rate for the *pair* airport+hour, via the same `add_history_feature`. Captures "ATL at 6pm" beyond "ATL" or "6pm" separately.

`DepHour` is added to `flights` first, because both features need the scheduled hour.

In [ ]:
flights["DepHour"] = (flights["CRSDepTime"] // 100).replace(24, 0)

feat["OriginHourCongestion"] = (
    flights.groupby(["Origin", "FlightDate", "DepHour"], observed=True)["DepHour"]
    .transform("size")
    .to_numpy(dtype="int32")
)


feat["OriginHourHist"] = add_history_feature(flights, ["Origin", "DepHour"])

feat[["OriginHourCongestion", "OriginHourHist"]].describe()

,OriginHourCongestion,OriginHourHist
count,6.965247e+06,6.965247e+06
mean,2.362590e+01,2.161583e-01
std,2.040283e+01,7.657205e-02
min,1.000000e+00,4.108448e-02
25%,7.000000e+00,1.595149e-01
50%,1.900000e+01,2.093332e-01
75%,3.300000e+01,2.651953e-01
max,1.000000e+02,5.344006e-01


## Step 4b - leakage guard

The single most valuable cell in this notebook: a test that fails loudly if a post-flight column slips into the features, or if the history ever peeks at the present/future. We (a) assert no `LEAKAGE` column is present, and (b) rebuild one carrier's history by hand and check the value matches sums from **strictly earlier days only**.

In [ ]:
assert not (set(feat.columns) & set(LEAKAGE)), "a leakage column reached the features!"

c = "9E"
sub = flights.loc[flights["Reporting_Airline"] == c, ["FlightDate", "y"]]
dates = np.sort(sub["FlightDate"].unique())

first_mask = (flights["Reporting_Airline"] == c) & (flights["FlightDate"] == dates[0])
assert np.allclose(feat.loc[first_mask, "CarrierHist"].unique(), np.float32(PRIOR))

d = dates[20]
past = sub.loc[sub["FlightDate"] < d, "y"]
expected = (past.sum() + M * PRIOR) / (len(past) + M)
got = feat.loc[(flights["Reporting_Airline"] == c) & (flights["FlightDate"] == d), "CarrierHist"].iloc[0]
assert np.isclose(got, expected, atol=1e-4), (got, expected)

print("leakage guard passed: no post-flight columns, and history uses only the past.")

leakage guard passed: no post-flight columns, and history uses only the past.


## Step 5 - final feature matrix

Assemble `X` (features) and `y` (label), and keep `FlightDate` **aside** - it drives the time-based split in S8 but is not a feature itself.

We deliberately keep **both** the raw categorical identity (`Origin`, `Dest`, `Reporting_Airline`) **and** its history rate: the identity captures a fixed effect, the history captures the shrunk, time-aware signal - they are complementary, not leaky (history is past-only). `Distance` and `DistanceGroup` stay redundant on purpose; XGBoost ignores the weaker split, so we let S10 importance decide rather than dropping blindly.

Note we do **not** impute missing values: XGBoost sends NaN down a learned default branch, which is more honest than filling with a mean.

In [ ]:
flight_date = flights["FlightDate"]

X = feat.drop(columns="y")
y = feat["y"]

print("X:", X.shape, "| y positive rate:", round(float(y.mean()), 4))
print(X.isna().sum().loc[lambda s: s > 0])
X.dtypes

X: (6965247, 20) | y positive rate: 0.2082

missing per column (XGBoost handles NaN natively - learns a default direction, so we do NOT impute):
Series([], dtype: int64)


Month                       int8
DayOfWeek                   int8
Reporting_Airline       category
Origin                  category
Dest                    category
OriginState             category
DestState               category
CRSDepTime                 int16
CRSArrTime                 int16
CRSElapsedTime           float32
Distance                 float32
DistanceGroup               int8
DepHour                    int16
IsWeekend                  int64
Season                  category
CarrierHist              float32
OriginHist               float32
RouteHist                float32
OriginHourCongestion       int32
OriginHourHist           float32
dtype: object

## Step 6 - integration check: the module must reproduce this notebook

Everything above is a prototype written to be read. The reusable version lives in
`flight_delay.transform.features`, and Phase B (Spark / Airflow) will wrap that module
instead of re-deriving the logic here. Two copies of the same logic drift apart
quietly, so this cell pins them together: if the module ever stops reproducing the
matrix built above, this is what notices.

`Season` is the one deliberate difference. This notebook casts it to `category` in
step 3, while `build_features` leaves it a plain string: that cast is part of the
contract with XGBoost (`enable_categorical=True`), not part of feature engineering,
so it belongs in S8. The check aligns that dtype so the comparison is about values
rather than about a decision already made.

In [ ]:
from flight_delay.transform import features as F

X_mod, y_mod, date_mod = F.build_features(flights)

# Align only the dtype we deliberately deferred to S8 (see above); everything else
# is compared strictly, dtypes included.
X_cmp = X_mod.copy()
X_cmp["Season"] = X_cmp["Season"].astype(X["Season"].dtype)

assert list(X_cmp.columns) == list(X.columns), "column set or order drifted"
pd.testing.assert_frame_equal(X_cmp, X, check_dtype=True)
assert (y_mod == y.to_numpy()).all(), "label drifted"
assert date_mod.equals(flight_date), "flight_date drifted"

print("build_features reproduces this notebook:", X_mod.shape)